# Machine learning for earth observation data processing pipeline Lukas Ripp

## Part 1: Training CNNs for Remote Sensing Classification

### Setup

In [1]:
import os

# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import lightning as pl
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

## Custom Dataset Class

In [2]:
class YourCustomDataset(Dataset):
    """
    Custom Dataset for Sentinel-2 patch classification.

    Parameters:
    -----------
    data : numpy.ndarray
        Patch data with shape (n_samples, height, width, channels)
    labels : numpy.ndarray
        Target labels with shape (n_samples,)
    """

    def __init__(self, data, labels):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]              # Shape: (H, W, C)
        x = x.permute(2, 0, 1)          # Convert to (C, H, W) for PyTorch
        y = self.labels[idx]
        return x, y

## Data Loading and Preprocessing

In [3]:
import numpy as np
from pathlib import Path

# --------------------------------------------------
# 1) Load train / val / test datasets
# --------------------------------------------------

BASE_DATA_DIR = Path("/p/scratch/training2600/ripp1/Project/training_data")

train_data = np.load(BASE_DATA_DIR / "train" / "patches_train.npz")
val_data = np.load(BASE_DATA_DIR / "val" / "patches_val.npz")
test_data = np.load(BASE_DATA_DIR / "test" / "patches_test.npz")

X_train = train_data["patches"]
y_train = np.load(BASE_DATA_DIR / "train" / "labels_train.npy")

X_val = val_data["patches"]
y_val = np.load(BASE_DATA_DIR / "val" / "labels_val.npy")

X_test = test_data["patches"]
y_test = np.load(BASE_DATA_DIR / "test" / "labels_test.npy")

print("Loaded datasets:")
print(f"  X_train: {X_train.shape}, dtype={X_train.dtype}")
print(f"  y_train: {y_train.shape}, dtype={y_train.dtype}")
print(f"  X_val:   {X_val.shape}, dtype={X_val.dtype}")
print(f"  y_val:   {y_val.shape}, dtype={y_val.dtype}")
print(f"  X_test:  {X_test.shape}, dtype={X_test.dtype}")
print(f"  y_test:  {y_test.shape}, dtype={y_test.dtype}")

# --------------------------------------------------
# 2) Basic preprocessing
# --------------------------------------------------
# Data is already normalized during patch extraction.
# Only ensure correct dtype here.

X_train = X_train.astype(np.float32)
X_val = X_val.astype(np.float32)
X_test = X_test.astype(np.float32)

print("\nPatch value ranges:")
print(f"  Train: [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"  Val:   [{X_val.min():.3f}, {X_val.max():.3f}]")
print(f"  Test:  [{X_test.min():.3f}, {X_test.max():.3f}]")

# Do NOT flatten for CNN training
print("\nCNN input shape per sample:")
print(f"  {X_train.shape[1:]} (H, W, C)")

# --------------------------------------------------
# 3) Remap labels to contiguous 0..K-1
# --------------------------------------------------
# Use all labels across splits so class indices stay consistent.

all_labels = np.concatenate([y_train, y_val, y_test])
unique_labels = np.unique(all_labels)

label_to_idx = {int(lbl): i for i, lbl in enumerate(unique_labels)}
idx_to_label = {i: int(lbl) for i, lbl in enumerate(unique_labels)}

y_train = np.array([label_to_idx[int(lbl)] for lbl in y_train], dtype=np.int64)
y_val = np.array([label_to_idx[int(lbl)] for lbl in y_val], dtype=np.int64)
y_test = np.array([label_to_idx[int(lbl)] for lbl in y_test], dtype=np.int64)

print(f"\nLabel remapping (original -> index):")
for orig, idx in label_to_idx.items():
    print(f"  CORINE {orig:2d} -> class index {idx}")
print(f"Total classes: {len(unique_labels)}")

# --------------------------------------------------
# 4) Dataset summary
# --------------------------------------------------

print(f"\nTraining samples:   {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Test samples:       {X_test.shape[0]}")
print(f"Patch shape:        {X_train.shape[1:]}")
print(f"Number of classes:  {len(unique_labels)}")

# --------------------------------------------------
# 5) Sanity check label distribution
# --------------------------------------------------

def label_hist(y_arr, name, topk=10):
    vals, cnts = np.unique(y_arr, return_counts=True)
    order = np.argsort(cnts)[::-1]
    print(f"\n{name} label distribution (top {topk}):")
    for v, c in zip(vals[order][:topk], cnts[order][:topk]):
        orig = idx_to_label[int(v)]
        print(f"  idx {int(v):2d} (CORINE {orig:2d}): {int(c)}")

label_hist(y_train, "Train")
label_hist(y_val, "Val")
label_hist(y_test, "Test")

Loaded datasets:
  X_train: (4000, 3, 3, 4), dtype=float32
  y_train: (4000,), dtype=uint8
  X_val:   (1000, 3, 3, 4), dtype=float32
  y_val:   (1000,), dtype=uint8
  X_test:  (1000, 3, 3, 4), dtype=float32
  y_test:  (1000,), dtype=uint8

Patch value ranges:
  Train: [0.000, 1.000]
  Val:   [0.000, 1.000]
  Test:  [0.000, 1.000]

CNN input shape per sample:
  (3, 3, 4) (H, W, C)

Label remapping (original -> index):
  CORINE  2 -> class index 0
  CORINE  3 -> class index 1
  CORINE  7 -> class index 2
  CORINE 11 -> class index 3
  CORINE 12 -> class index 4
  CORINE 18 -> class index 5
  CORINE 20 -> class index 6
  CORINE 23 -> class index 7
  CORINE 24 -> class index 8
Total classes: 9

Training samples:   4000
Validation samples: 1000
Test samples:       1000
Patch shape:        (3, 3, 4)
Number of classes:  9

Train label distribution (top 10):
  idx  5 (CORINE 18): 2184
  idx  7 (CORINE 23): 768
  idx  0 (CORINE  2): 524
  idx  4 (CORINE 12): 244
  idx  8 (CORINE 24): 104
  idx 

## Understanding Class Imbalance & Sampling Strategies

In [4]:
import numpy as np
import torch
from torch.utils.data import WeightedRandomSampler

# Convert training labels to numpy
y_train_np = np.asarray(y_train, dtype=np.int64)
classes, counts = np.unique(y_train_np, return_counts=True)

print("=" * 70)
print("TRAINING SET CLASS DISTRIBUTION")
print("=" * 70)
print(f"Total samples: {len(y_train_np)}")
print(f"Number of classes: {len(classes)}\n")

# Sort classes by frequency (descending)
sort_idx = np.argsort(-counts)

print("Classes (sorted by frequency):")
print("Index | CORINE | Count  | Percentage | Balance")
print("-" * 60)

for idx in sort_idx:
    c = classes[idx]
    n = counts[idx]
    pct = 100 * n / len(y_train_np)
    bar = "█" * int(pct / 2)

    print(
        f"{int(c):5d} | {idx_to_label[int(c)]:6d} | {int(n):6d} | {pct:6.1f}%  | {bar}"
    )

majority_class_pct = 100 * counts.max() / len(y_train_np)
print(f"\n⚠️ Majority class represents {majority_class_pct:.1f}% of the training data")
print("   This is the baseline accuracy for a naive classifier.")

# Compute imbalance ratio
imbalance_ratio = counts.max() / counts.min()
print(f"   Imbalance ratio (max/min): {imbalance_ratio:.1f}x")

TRAINING SET CLASS DISTRIBUTION
Total samples: 4000
Number of classes: 9

Classes (sorted by frequency):
Index | CORINE | Count  | Percentage | Balance
------------------------------------------------------------
    5 |     18 |   2184 |   54.6%  | ███████████████████████████
    7 |     23 |    768 |   19.2%  | █████████
    0 |      2 |    524 |   13.1%  | ██████
    4 |     12 |    244 |    6.1%  | ███
    8 |     24 |    104 |    2.6%  | █
    2 |      7 |     68 |    1.7%  | 
    1 |      3 |     54 |    1.4%  | 
    3 |     11 |     40 |    1.0%  | 
    6 |     20 |     14 |    0.3%  | 

⚠️ Majority class represents 54.6% of the training data
   This is the baseline accuracy for a naive classifier.
   Imbalance ratio (max/min): 156.0x


## Setting up the Model

In [5]:
import lightning.pytorch as pl
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvNet(pl.LightningModule):
    """
    Simple CNN classifier for Sentinel-2 patch classification.

    Expected input shape:
        (B, C, H, W)

    For this project:
        C = 4  (B02, B03, B04, B08)
        H = W = 3
    """

    def __init__(
        self,
        num_classes,
        in_channels=4,
        patch_size=3,
        lr=3e-4,
        max_epochs=60,
        class_weights=None,
    ):
        super().__init__()
        self.lr = lr
        self.in_channels = in_channels
        self.patch_size = patch_size
        self.num_classes = num_classes
        self.max_epochs = max_epochs

        # Feature extraction layers
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def forward(self, x):
        x = x.float()
        x = self.features(x)
        return self.classifier(x)

    def _loss(self, logits, y):
        return F.cross_entropy(logits, y, weight=self.class_weights)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()

        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("train_acc", acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()

        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("val_acc", acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()

        self.log("test_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("test_acc", acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.max_epochs)
        return {"optimizer": optimizer, "lr_scheduler": scheduler}

In [6]:
# ============================================================================
# CONFIGURATION: Choose Your Sampling Strategy
# ============================================================================

batch_size = 256
max_epochs = 100

print("\n" + "=" * 70)
print("SAMPLING STRATEGY COMPARISON")
print("=" * 70)

# Strategy 1: Inverse-frequency weights for weighted loss
print("\n1️⃣  STRATEGY 1: Class-Weighted Loss (No Resampling)")
print("-" * 70)
print("   How it works:")
print("   • Each class gets a weight = 1 / frequency")
print("   • Model sees all samples in their original distribution")
print("   • Loss contribution per class is equalized")
print("   • Pros: Simple, no data duplication")
print("   • Cons: Model still sees majority classes much more often")

# Compute sqrt-scaled weights (softens extreme weights)
raw_weights = np.zeros(len(classes), dtype=np.float64)
for c, n in zip(classes, counts):
    raw_weights[int(c)] = 1.0 / n

sqrt_weights = np.sqrt(raw_weights)
sqrt_weights /= sqrt_weights.sum() / len(classes)  # Normalize to mean=1

print("\n   Class weights (sqrt-scaled):")
print("   Index | CORINE | Weight")
print("   " + "-" * 35)
sort_idx = np.argsort(-sqrt_weights)
for c_idx in sort_idx:
    orig = idx_to_label[c_idx]
    print(f"   {c_idx:5d} | {orig:6d} | {sqrt_weights[c_idx]:6.2f}")

class_weights_t = torch.tensor(sqrt_weights, dtype=torch.float32)

# Strategy 2: Weighted Random Sampler
print("\n\n2️⃣  STRATEGY 2: Weighted Random Sampler (Resampling)")
print("-" * 70)
print("   How it works:")
print("   • Each sample gets weight = 1 / class_frequency")
print("   • Sampler draws batches with replacement")
print("   • Rare classes appear more often in mini-batches")
print("   • Pros: Direct oversampling, minorities are seen more often")
print("   • Cons: Duplication, may overfit on very rare classes")

# Map each training sample to its class weight
class_weight_by_value = {int(c): float(1.0 / n) for c, n in zip(classes, counts)}
sample_weights = np.array(
    [class_weight_by_value[int(lbl)] for lbl in y_train_np], dtype=np.float64
)

sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

print("   ✓ Sampler created")

# ============================================================================
# SELECT WHICH STRATEGY TO USE
# ============================================================================
print("\n\n" + "=" * 70)
print("SELECTED STRATEGY")
print("=" * 70)

# Recommended for this dataset because class imbalance is very strong
strategy = "class_weights"

if strategy == "class_weights":
    print(f"\n✅ Selected: CLASS-WEIGHTED LOSS")
    class_weights_for_loss = class_weights_t
    train_sampler = None
    train_shuffle = True
    print("   • Using class weights in the loss function")
    print("   • DataLoader will shuffle normally")

elif strategy == "weighted_sampler":
    print(f"\n✅ Selected: WEIGHTED RANDOM SAMPLER")
    class_weights_for_loss = None
    train_sampler = sampler
    train_shuffle = False
    print("   • Using weighted resampling in the training DataLoader")
    print("   • Rare classes will appear more often in mini-batches")

else:
    print(f"\n✅ Selected: NO IMBALANCE HANDLING")
    class_weights_for_loss = None
    train_sampler = None
    train_shuffle = True
    print("   • DataLoader will shuffle normally")

print("\n" + "=" * 70)
print("MODEL SETUP")
print("=" * 70)

# Auto-detect architecture parameters from patch data
num_classes = len(np.unique(y_train_np))
patch_size = X_train.shape[1]     # X_train shape: (N, H, W, C)
in_channels = X_train.shape[3]    # number of spectral bands

print(f"\n  num_classes    = {num_classes}")
print(f"  patch_size     = {patch_size}×{patch_size}")
print(f"  in_channels    = {in_channels}")
print(f"  batch_size     = {batch_size}")

# Create model
model = ConvNet(
    lr=3e-4,
    num_classes=num_classes,
    in_channels=in_channels,
    patch_size=patch_size,
    class_weights=class_weights_for_loss,
    max_epochs=max_epochs,
)

print(f"\n✓ CNN model created")

# Create datasets
train_dataset = YourCustomDataset(X_train, y_train)
val_dataset = YourCustomDataset(X_val, y_val)
test_dataset = YourCustomDataset(X_test, y_test)

# Create dataloaders
train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=train_sampler,
    shuffle=train_shuffle,
    num_workers=2,
    pin_memory=True,
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print("✓ DataLoaders created")
print(f"  • Training batches:   {len(train_dataloader)}")
print(f"  • Validation batches: {len(val_dataloader)}")
print(f"  • Test batches:       {len(test_dataloader)}")


SAMPLING STRATEGY COMPARISON

1️⃣  STRATEGY 1: Class-Weighted Loss (No Resampling)
----------------------------------------------------------------------
   How it works:
   • Each class gets a weight = 1 / frequency
   • Model sees all samples in their original distribution
   • Loss contribution per class is equalized
   • Pros: Simple, no data duplication
   • Cons: Model still sees majority classes much more often

   Class weights (sqrt-scaled):
   Index | CORINE | Weight
   -----------------------------------
       6 |     20 |   2.54
       3 |     11 |   1.50
       1 |      3 |   1.29
       2 |      7 |   1.15
       8 |     24 |   0.93
       4 |     12 |   0.61
       0 |      2 |   0.42
       7 |     23 |   0.34
       5 |     18 |   0.20


2️⃣  STRATEGY 2: Weighted Random Sampler (Resampling)
----------------------------------------------------------------------
   How it works:
   • Each sample gets weight = 1 / class_frequency
   • Sampler draws batches with replacem

## Training Model

In [7]:
# Verify data shapes before training
xb, yb = next(iter(train_dataloader))
print("Quick data check:")
print(f"  Batch X shape: {xb.shape} (dtype={xb.dtype})")
print(f"  Batch y shape: {yb.shape} (dtype={yb.dtype})")
print(f"  y range:       [{yb.min().item()}, {yb.max().item()}]")
print(f"  Unique classes in batch: {sorted(torch.unique(yb).tolist())}")

Quick data check:
  Batch X shape: torch.Size([256, 4, 3, 3]) (dtype=torch.float32)
  Batch y shape: torch.Size([256]) (dtype=torch.int64)
  y range:       [0, 8]
  Unique classes in batch: [0, 1, 2, 3, 4, 5, 6, 7, 8]


In [8]:
print("\n" + "=" * 70)
print("STARTING TRAINING")
print("=" * 70)
print(f"Strategy:     {strategy}")
print(f"Max epochs:   {max_epochs}")
print(f"Batch size:   {batch_size}")
print("=" * 70 + "\n")

trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=max_epochs,
    gradient_clip_val=1.0,
    log_every_n_steps=1,
    enable_progress_bar=True,
)

# Start training
trainer.fit(model, train_dataloader, val_dataloader)

print("\n" + "=" * 70)
print("TESTING")
print("=" * 70)
trainer.test(model, dataloaders=test_dataloader)


STARTING TRAINING
Strategy:     class_weights
Max epochs:   100
Batch size:   256



GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3     ]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ features   │ Sequential │ 76.6 K │ train │     0 │
│ 1 │ classifier │ Sequential │  8.8 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 85.4 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 85.4 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 14                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

SLURM auto-requeueing enabled. Setting signal handlers.


...

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3     ]
SLURM auto-requeueing enabled. Setting signal handlers.



TESTING


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.42399999499320984    │
│         test_loss         │    1.5838050842285156     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 1.5838050842285156, 'test_acc': 0.42399999499320984}]

In [9]:
# ============================================================
# Lab 7 evaluation functions
# ============================================================

import torch
from torch.utils.data import DataLoader, TensorDataset

# X_test, y_test müssen existieren
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Falls (N, H, W, C) → umwandeln
if X_test_tensor.shape[-1] == 4:
    X_test_tensor = X_test_tensor.permute(0, 3, 1, 2)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

In [10]:
model.eval()
device = next(model.parameters()).device

all_preds = []
all_targets = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = model(xb)

        preds = torch.argmax(logits, dim=1).cpu().numpy()
        targets = yb.numpy()

        all_preds.append(preds)
        all_targets.append(targets)

y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_targets)

In [11]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)

In [16]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.clf()
plt.figure(figsize=(8,6))

sns.heatmap(cm, annot=True, cmap="Blues")

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix base model")

plt.tight_layout()
plt.savefig("/p/project1/training2600/ripp1/results/Project/evaluation_base_pipeline/confusion_matrix.png", dpi=300)
plt.show()